# Capítulo 1: Bandidos Multi-Brazo (Multi-Armed Bandits) y la Política $\varepsilon$-Greedy

En este cuaderno implementamos el problema arquetípico de **Bandidos Multi-Brazo** para estudiar el dilema entre **exploración** y **explotación**.

OptiBot debe seleccionar entre $k=4$ tolvas de clasificación de paquetes con rendimientos promedio desconocidos.
Utiliza una política $\varepsilon$-greedy con estimación incremental de medias:
$$Q_{t+1}(a) = Q_t(a) + \frac{1}{N(a)} \big[ R_t - Q_t(a) \big]$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class BandidoAlmacen:
    def __init__(self, k=4, epsilon=0.10):
        self.k = k
        self.epsilon = epsilon
        # Valores objetivos subyacentes del entorno
        self.q_real = np.array([3.2, 4.8, 2.1, 4.1])
        self.Q = np.zeros(k) # Vector de estimaciones iniciales
        self.N = np.zeros(k) # Vector de frecuencias de selección

    def seleccionar_accion(self):
        # Selección epsilon-greedy
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k) # Exploración uniforme
        return int(np.argmax(self.Q))         # Explotación codiciosa

    def ejecutar_paso(self, accion):
        # Simulación de respuesta estocástica con ruido gaussiano
        recompensa = np.random.normal(loc=self.q_real[accion], scale=1.0)
        self.N[accion] += 1
        # Actualización mediante la regla incremental
        error = recompensa - self.Q[accion]
        self.Q[accion] += (1.0 / self.N[accion]) * error
        return recompensa

# Simulación de 2,000 despachos en el almacén
np.random.seed(42)
robot = BandidoAlmacen(k=4, epsilon=0.10)
historial_recompensas = []

for t in range(2000):
    a = robot.seleccionar_accion()
    r = robot.ejecutar_paso(a)
    historial_recompensas.append(r)

print("Valores Reales Subyacentes:", robot.q_real)
print("Estimaciones Aprendidas:   ", np.round(robot.Q, 2))
print("Frecuencia de Selección:   ", robot.N)


## Comparación Gráfica: Valores Reales vs Aprendidos y Frecuencia de Acciones


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

acciones = [f"Tolva {i+1}" for i in range(robot.k)]
x = np.arange(robot.k)
width = 0.35

# Comparación Q vs q*
ax1.bar(x - width/2, robot.q_real, width, label='Valor Real q*(a)', color='#2ca02c')
ax1.bar(x + width/2, robot.Q, width, label='Estimado Q(a)', color='#1f77b4')
ax1.set_xticks(x)
ax1.set_xticklabels(acciones)
ax1.set_title('Convergencia de Estimaciones Q')
ax1.set_ylabel('Recompensa Esperada')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Frecuencia de selección
ax2.bar(acciones, robot.N, color='#ff7f0e')
ax2.set_title('Frecuencia de Selección de Acciones (2,000 pasos)')
ax2.set_ylabel('Número de Veces Elegida')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
